# Member 5 · Frozen SBERT + TextCNN
YU Guanlin · DSAI4205

普通 Jupyter Notebook。默认从已提交的逐条预测重算结果，不需要 GPU、嵌入缓存或云端账号。
重新训练需在最后一节明确开启；数据和固定分组保持团队版本。

In [ ]:
import sys, os, json, subprocess
from pathlib import Path
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/config.py').is_file()),None)
if ROOT is None:raise RuntimeError('请从仓库根目录或 Member5_file 内打开 Notebook')
sys.path.insert(0,str(ROOT));sys.path.insert(0,str(ROOT/'Member5_file'))
import pandas as pd
from IPython.display import display,Image,Markdown
from release import verify_evidence
EVIDENCE=ROOT/'Member5_file/evidence/local_refinement_20261113'
print('Project:',ROOT.name)

## 已完成的任务
冻结多语言 SBERT 生成句向量和 token 序列，后接独立 TextCNN；另有按折训练的 FastText 对照。
最新实验完成六配置筛选、分组嵌套验证及三个随机种子。保留集只输出预测，不评分。

In [ ]:
verification=verify_evidence(EVIDENCE)
print('Evidence verification passed')
print('Development rows:',verification['development_rows'],'| Reserved split rows:',verification['reserved_rows'])
print(verification['scope'])
display(pd.DataFrame(verification['original_mvp']))

## 最近一次完整实验
下表是本地实验的选模流程 OOF，并非固定最终模型的独立测试成绩。
最新选模流程未超过同协议 C0，因此作为首版研究记录交付，不宣称优化成功。

In [ ]:
table=pd.read_csv(EVIDENCE/'comparison.csv')
display(table[['model','seed','accuracy','macro_precision','macro_recall','macro_f1']])
summary=json.loads((EVIDENCE/'conclusion.json').read_text())
display(pd.DataFrame([{'model':'Selection pipeline','mean_macro_f1':summary['selected_mean']},
                      {'model':'Same-protocol C0','mean_macro_f1':summary['control_mean']},
                      {'model':'Historical official baseline','mean_macro_f1':summary['baseline_macro_f1']}]))
print('Mean delta:',round(summary['mean_delta'],4),'| Practical target met:',summary['practical_target_met'])
display(pd.DataFrame(verification['class_tradeoffs']))

In [ ]:
for name in ['learning_curves.png','epoch_losses.png','confusion_selected.png','seed_comparison.png']:
    display(Image(filename=str(EVIDENCE/name)))

## 结果的边界
历史 OOF 和团队保留集曾被接触；这些是追加开发证据。训练／验证差距支持检查泛化问题，但不能唯一归因于模型容量或编码器。
上方核验重算公开预测和 bootstrap；历史完整训练、分组、源码及权重重载检查见 independent_verification.json。
本发布版清除了云端执行代码，其新运行签名与历史训练版不同；不能把历史权重审计当成对修改后源码的重新训练证明。

## 可选：在当前 Jupyter 环境复现
先按 README 安装依赖，并在本机放好冻结缓存。初次提取 SBERT 请按 README 的隔离编码命令操作；首次模型下载需要联网。
保留 RUN_TRAINING=False 可反复查看结果。开启后使用当前内核的 Python、自动选择可用 CPU／MPS／CUDA，并保存本机日志及断点。
完整训练有两小时累计预算；不要通过换编号绕过预算。运行参数、每次尝试和失败均保留。

In [ ]:
RUN_TRAINING=False
RUN_ID='m5_jupyter_first_release'
DEVICE='auto'
if RUN_TRAINING:
    cache=Path(os.environ.get('DATA_ROOT',ROOT/'data'))/'emb'
    manifest=json.loads((ROOT/'Member5_file/evidence/embedding_manifest.json').read_text())
    missing=[name for name in manifest['files'] if not (cache/name).is_file()]
    if missing:raise FileNotFoundError('缺少冻结缓存：'+', '.join(missing)+'。请先按 README 的嵌入准备步骤操作。')
    subprocess.run([sys.executable,'-u',str(ROOT/'Member5_file/run_local.py'),
                    '--device',DEVICE,'--run-id',RUN_ID],cwd=ROOT,check=True)
else:
    print('Review mode: no training or downloads were started.')